# Initialization Ablation: Random vs Graph Eigenvector

**Goal:** Show that graph structure improves embeddings

**Comparison:**
- Random-Init: Standard random initialization
- Graph-Init: Eigenvector-based initialization from graph Laplacian

**Settings:**
- `min_edges=5000`: Only ~10 densest relations (fast eigendecomp)
- `num_eigenvectors=3`: Minimal eigenvectors (fast & stable)

**Hypothesis:** Graph eigenvector initialization captures connectivity patterns

In [1]:
# Setup
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

Cloning into '/content/kg-bayesian-prior'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (291/291), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 291 (delta 180), reused 199 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (291/291), 232.14 KiB | 3.68 MiB/s, done.
Resolving deltas: 100% (180/180), done.


In [2]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Optimized config - minimal eigendecomp
CONFIG = {
    'embedding_dim': 50,
    'epochs': 10,
    'num_inducing': 200,
    'mrr_sample': 500,
    'min_edges': 5000,       # Only ~10 dense relations
    'num_eigenvectors': 3,   # Minimal eigenvectors
}
print(f"Config: {CONFIG}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

results = {}

Device: cpu
Config: {'embedding_dim': 50, 'epochs': 10, 'num_inducing': 200, 'mrr_sample': 500, 'min_edges': 5000, 'num_eigenvectors': 3}
FB15k-237 not found. Downloading...


train.txt: 21.0MB [00:00, 86.3MB/s]


valid.txt: 1.29MB [00:00, 8.16MB/s]


test.txt: 1.51MB [00:00, 10.8MB/s]


FB15k-237 download complete!
Data: 272,115 train, 20,466 test


In [3]:
def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def evaluate(model, name):
    """Fast evaluation with small samples"""
    print(f"\nEvaluating {name}...")
    model.eval()

    # MRR (small sample)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 50), desc="MRR", leave=False):
            batch = sample[i:i+50]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE (small sample)
    ece_sample = 500
    ece_idx = np.random.choice(len(test_data), ece_sample, replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    with torch.no_grad():
        h, r, t = [torch.tensor(all_t[:,j], device=device) for j in range(3)]
        scores = model.score_triple(h, r, t)
        conf = torch.sigmoid(scores).cpu().numpy()
    ece, _ = expected_calibration_error(conf, labels)

    # AUROC (small sample)
    ood_sample = 500
    id_idx = np.random.choice(len(test_data), ood_sample, replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", ood_sample)

    def get_unc(triples):
        with torch.no_grad():
            h, r, t = [torch.tensor(triples[:,j], device=device) for j in range(3)]
            pred = model.predict_with_uncertainty(h, r, t)
            return pred['total'].cpu().numpy()

    auroc = compute_auroc(get_unc(id_triples), get_unc(ood_triples))

    results[name] = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "auroc": auroc}
    print(f"{name}: MRR={mrr:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return results[name]

---
## GP-KGE (RBF Kernel) - Identity Prior
---

In [4]:
print("="*50 + "\nGP-KGE (Random-Init)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_rbf = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="rbf",
    num_inducing=CONFIG['num_inducing']
).to(device)

print("Using random initialization")

opt = torch.optim.Adam(model_rbf.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (Random-Init)")):
    model_rbf.train()
    loss_sum, n = 0, 0

    for batch_idx, st in enumerate(range(0, len(train_data), 1024)):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_rbf.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_rbf.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rbf.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_rbf, "GP-KGE (Random-Init)")
del model_rbf
clear_mem()

GP-KGE (Random-Init)
Using random initialization


GP-KGE (Random-Init):   0%|          | 0/10 [00:00<?, ?it/s]

Training time: 37.2s

Evaluating GP-KGE (Random-Init)...


MRR:   0%|          | 0/10 [00:00<?, ?it/s]

GP-KGE (Random-Init): MRR=0.2224, H@10=0.3220, ECE=0.0994, AUROC=0.8702


---
## GP-KGE (Relation-Aware Kernel) - Graph Prior
---

In [5]:
print("="*50 + "\nGP-KGE (Graph-Init)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_graph = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=CONFIG['num_inducing']
).to(device)

# Graph-based init with minimal eigendecomp
print("Setting graph structure (minimal eigendecomp)...")
model_graph.set_graph(
    train_data,
    num_eigenvectors=CONFIG['num_eigenvectors'],  # Only 3 eigenvectors
    min_edges=CONFIG['min_edges'],                 # Only ~10 relations
    show_progress=True,
    init_embeddings=True  # Use eigenvectors for initialization
)
print("Graph initialization complete")

GP-KGE (Graph-Init)
Setting graph structure (minimal eigendecomp)...
Computing spectral decomposition for 13/237 relations...
  (skipping 224 sparse relations with <5000 edges)


Eigendecomp: 100%|██████████| 13/13 [00:50<00:00,  3.90s/rel]

Spectral decomposition complete: 12/13 relations succeeded
Initialized embeddings from 12 relation eigenvectors
Graph initialization complete


In [6]:
# Training (BCE only, no KL for speed)
opt = torch.optim.Adam(model_graph.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (Graph-Init)")):
    model_graph.train()
    loss_sum, n = 0, 0

    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_graph.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_graph.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_graph.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_graph, "GP-KGE (Graph-Init)")
del model_graph
clear_mem()

GP-KGE (Graph-Init):   0%|          | 0/10 [00:00<?, ?it/s]

Training time: 83.8s

Evaluating GP-KGE (Graph-Init)...


MRR:   0%|          | 0/10 [00:00<?, ?it/s]

GP-KGE (Graph-Init): MRR=0.2091, H@10=0.3320, ECE=0.0552, AUROC=0.8767


---
## Results
---

In [7]:
print("\n" + "="*70)
print("KERNEL ABLATION RESULTS")
print("="*70)
print(f"{'Model':<25} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'AUROC':>8}")
print("-"*70)

for name, r in results.items():
    print(f"{name:<25} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f}")

if len(results) == 2:
    rand = results.get("GP-KGE (Random-Init)", {})
    graph = results.get("GP-KGE (Graph-Init)", {})
    if rand and graph:
        print("\n" + "="*70)
        print("IMPROVEMENT (Graph-Init vs Random-Init)")
        print("="*70)
        print(f"  MRR:   {rand['mrr']:.4f} -> {graph['mrr']:.4f} ({(graph['mrr']-rand['mrr'])/rand['mrr']*100:+.1f}%)")
        print(f"  H@10:  {rand['hits@10']:.4f} -> {graph['hits@10']:.4f} ({(graph['hits@10']-rand['hits@10'])/rand['hits@10']*100:+.1f}%)")
        print(f"  AUROC: {rand['auroc']:.4f} -> {graph['auroc']:.4f} ({(graph['auroc']-rand['auroc'])/rand['auroc']*100:+.1f}%)")
        print(f"  ECE:   {rand['ece']:.4f} -> {graph['ece']:.4f} ({(rand['ece']-graph['ece'])/rand['ece']*100:+.1f}% better)")


KERNEL ABLATION RESULTS
Model                          MRR      H@1     H@10      ECE    AUROC
----------------------------------------------------------------------
GP-KGE (Random-Init)        0.2224   0.1680   0.3220   0.0994   0.8702
GP-KGE (Graph-Init)         0.2091   0.1420   0.3320   0.0552   0.8767

IMPROVEMENT (Graph-Init vs Random-Init)
  MRR:   0.2224 -> 0.2091 (-6.0%)
  H@10:  0.3220 -> 0.3320 (+3.1%)
  AUROC: 0.8702 -> 0.8767 (+0.7%)
  ECE:   0.0994 -> 0.0552 (+44.4% better)


In [8]:
# Save
with open('kernel_ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to kernel_ablation_results.json")

try:
    from google.colab import files
    files.download('kernel_ablation_results.json')
except:
    pass

Saved to kernel_ablation_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>